# Chapter 03: Multi-Task HydraNet Perception & Uncertainty Loss

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/03_hydranet_multitask_learning.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How do we train 15 perception heads on a single shared backbone without destructive gradient interference?*

---

## 1. 🚨 The Real-World Dilemma
Running 15 separate networks draws kilowatts of power. A shared HydraNet solves compute, but naive loss summation causes task competition (Negative Transfer). Kendall Uncertainty Weighting learns task observation noise $\sigma_i$ to dynamically balance gradients.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

class KendallUncertaintyLoss(nn.Module):
    def __init__(self, num_tasks=3):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, losses):
        total_loss = 0.0
        for i, loss in enumerate(losses):
            s = self.log_vars[i]
            precision = torch.exp(-s)
            total_loss += 0.5 * precision * loss + 0.5 * s
        return total_loss

loss_fn = KendallUncertaintyLoss(3)
optimizer = torch.optim.Adam(loss_fn.parameters(), lr=0.05)
task_losses = [torch.tensor(0.2), torch.tensor(3.5), torch.tensor(0.05)]

history = []
for _ in range(50):
    optimizer.zero_grad()
    loss = loss_fn(task_losses)
    loss.backward()
    optimizer.step()
    history.append(loss_fn.log_vars.detach().clone())

plt.figure(figsize=(8, 3.5))
plt.plot([h[0].item() for h in history], label="Bbox (0.2)", lw=2)
plt.plot([h[1].item() for h in history], label="Drivable (3.5)", lw=2)
plt.plot([h[2].item() for h in history], label="Lights (0.05)", lw=2)
plt.title("Kendall Learnable Log-Variances s = log(sigma^2) Converging")
plt.xlabel("Step")
plt.ylabel("Learned Log-Variance s")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **New task degrades existing tasks by >10%** | Negative transfer / conflicting gradients ($\mathbf{g}_1 \cdot \mathbf{g}_2 < 0$). | Compute cosine similarity between head gradients. | Use Kendall Uncertainty weighting or PCGrad gradient projection. |
| **One head's variance explodes ($s \to \infty$)** | Model cheats by declaring a difficult task 'infinite noise' to ignore it. | Check `log_vars` values during training. | Clamp max log-variance ($s_i \le 3.0$). |